# Atmospheric Transmission Estimation
Written by Kiyoaki Okudaira<br>
*Kyushu University Hanada Lab / University of Washington / IAU CPS SatHub<br>
(okudaira.kiyoaki.528@s.kyushu-u.ac.jp or kiyoaki@uw.edu)<br>
<br>
Estimate atmospheric transmission from multiple magzero with different altitudes.<br>
<br>
**History**<br>
coding 2026-07-22 : 1st coding<br>
<br>
(c) 2026 Kiyoaki Okudaira - Kyushu University Hanada Lab (SSDL) / University of Washington / IAU CPS SatHub

### Import and initial settings
**PATH settings**

In [ ]:
# Machine-specific roots (PATH_images, PATH_input, PATH_output, PATH_work)
# come from satphot_paths.py / paths.json.
from satphot_paths import *

# MAGZERO JSON files measured at different altitudes (created by magzero.ipynb)
MAGZERO_FILES = [
    PATH_output / "magzero" / name
    for name in [
        "magzero_2026-07-21T130630_GAIN56.json",
        "magzero_2026-07-21T130659_GAIN56.json",
        "magzero_2026-07-21T130730_GAIN56.json",
        "magzero_2026-07-21T130801_GAIN56.json",
        "magzero_2026-07-21T130825_GAIN56.json",
        "magzero_2026-07-21T130916_GAIN56.json",
        "magzero_2026-07-21T131001_GAIN56.json",
        "magzero_2026-07-21T131039_GAIN56.json",
        "magzero_2026-07-21T131117_GAIN56.json",
        "magzero_2026-07-21T131212_GAIN56.json",
    ]
]

# Output PATH
OUTPUT_DIRECTORY = PATH_output / "airmass"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Observatory location
OBSERVATORY_LATITUDE_DEG = 33.598889
OBSERVATORY_LONGITUDE_DEG = 130.211667
OBSERVATORY_ALTITUDE_M = 73.0

# Zeropoint reference
ZEROPOINT_KEY = "photometric_zeropoint_elec"

# If true: use zeropoint scatter
USE_SCATTER_WEIGHTS = False

**Standard libraries**

In [ ]:
from __future__ import annotations

import json
import math
from typing import Any, Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time
import astropy.units as u


### Functions

In [ ]:
RA_HEADER_CANDIDATES = ("CRVAL1", "_RVAL1")
DEC_HEADER_CANDIDATES = ("CRVAL2", "_RVAL2")
TIME_HEADER_CANDIDATES = ("DATE-AVG", "DATE-OBS", "JD_UTC")


def observation_value(data: dict[str, Any], key: str) -> Any:
    """MAGZERO JSONの observation[key]['value'] を取得する。"""
    entry = data.get("observation", {}).get(key)
    if isinstance(entry, dict):
        return entry.get("value")
    return entry


def first_available_observation_value(
    data: dict[str, Any],
    candidates: Iterable[str],
) -> tuple[str, Any]:
    for key in candidates:
        value = observation_value(data, key)
        if value is not None:
            return key, value

    raise KeyError(
        "必要な観測ヘッダーが見つかりません: "
        + ", ".join(candidates)
    )


def parse_observation_time(data: dict[str, Any]) -> Time:
    """DATE-AVG、DATE-OBS、JD_UTCの順で観測時刻を読み取る。"""
    key, value = first_available_observation_value(
        data,
        TIME_HEADER_CANDIDATES,
    )

    if key == "JD_UTC":
        return Time(float(value), format="jd", scale="utc")

    return Time(str(value), format="isot", scale="utc")


def load_magzero_measurement(
    path: Path,
    location: EarthLocation,
    zeropoint_key: str,
) -> dict[str, Any]:
    """1つのMAGZERO JSONから必要な観測量を読み取る。"""
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    phot_params = data.get("phot_params", {})
    if zeropoint_key not in phot_params:
        raise KeyError(
            f"{path.name}: phot_paramsに{zeropoint_key!r}がありません。"
        )

    _, ra_value = first_available_observation_value(
        data,
        RA_HEADER_CANDIDATES,
    )
    _, dec_value = first_available_observation_value(
        data,
        DEC_HEADER_CANDIDATES,
    )

    observation_time = parse_observation_time(data)

    target = SkyCoord(
        ra=float(ra_value) * u.deg,
        dec=float(dec_value) * u.deg,
        frame="icrs",
    )

    altaz = target.transform_to(
        AltAz(
            obstime=observation_time,
            location=location,
        )
    )

    altitude_deg = float(altaz.alt.deg)
    if altitude_deg <= 0:
        raise ValueError(
            f"{path.name}: 仰角が{altitude_deg:.3f} degで、地平線下です。"
        )

    zenith_angle_deg = 90.0 - altitude_deg

    # 平面大気近似
    # X = 1/sin(仰角) = 1/cos(天頂角)
    airmass_plane_parallel = 1.0 / math.sin(
        math.radians(altitude_deg)
    )

    scatter = phot_params.get("zeropoint_scatter")

    return {
        "file": path.name,
        "object_name": observation_value(data, "OBJECT"),
        "observation_time_utc": observation_time.utc.isot,
        "ra_deg": float(ra_value),
        "dec_deg": float(dec_value),
        "altitude_deg": altitude_deg,
        "zenith_angle_deg": zenith_angle_deg,
        "airmass_plane_parallel": airmass_plane_parallel,
        "airmass_astropy_secz": float(altaz.secz.value),
        "zeropoint": float(phot_params[zeropoint_key]),
        "zeropoint_scatter": (
            np.nan if scatter is None else float(scatter)
        ),
        "exposure_s": observation_value(data, "EXPTIME"),
        "egain_e_per_adu": (
            observation_value(data, "EGAINSAV")
            if observation_value(data, "EGAINSAV") is not None
            else observation_value(data, "EGAIN")
        ),
    }


def fit_zeropoint_vs_airmass(
    table: pd.DataFrame,
    use_scatter_weights: bool = True,
) -> dict[str, float]:
    """
    ZP = intercept + slope * X をフィットする。

    slope = 2.5 log10(p)
    p = 10**(slope/2.5)
    """
    x = table["airmass_plane_parallel"].to_numpy(dtype=float)
    y = table["zeropoint"].to_numpy(dtype=float)
    sigma = table["zeropoint_scatter"].to_numpy(dtype=float)

    if len(x) < 2:
        raise ValueError("2つ以上のMAGZEROファイルが必要です。")

    if np.ptp(x) <= 0:
        raise ValueError("すべての観測でairmassが同じです。")

    can_weight = (
        use_scatter_weights
        and np.all(np.isfinite(sigma))
        and np.all(sigma > 0)
    )

    if can_weight and len(x) > 2:
        coefficients, covariance = np.polyfit(
            x,
            y,
            deg=1,
            w=1.0 / sigma,
            cov=True,
        )
    elif len(x) > 2:
        coefficients, covariance = np.polyfit(
            x,
            y,
            deg=1,
            cov=True,
        )
    else:
        coefficients = np.polyfit(x, y, deg=1)
        covariance = np.full((2, 2), np.nan)

    slope = float(coefficients[0])
    intercept = float(coefficients[1])

    fitted = intercept + slope * x
    residuals = y - fitted

    slope_std = (
        float(np.sqrt(covariance[0, 0]))
        if np.isfinite(covariance[0, 0])
        else np.nan
    )

    transmission = 10.0 ** (slope / 2.5)
    extinction_mag_per_airmass = -slope

    transmission_std = (
        transmission * math.log(10.0) / 2.5 * slope_std
        if np.isfinite(slope_std)
        else np.nan
    )

    return {
        "slope_mag_per_airmass": slope,
        "slope_std_mag_per_airmass": slope_std,
        "zenith_zeropoint": intercept,
        "transmission_per_airmass": transmission,
        "transmission_std": transmission_std,
        "extinction_mag_per_airmass": extinction_mag_per_airmass,
        "rmse_mag": float(np.sqrt(np.mean(residuals**2))),
    }


### Read MAGZERO files

In [ ]:
location = EarthLocation.from_geodetic(
    lon=OBSERVATORY_LONGITUDE_DEG * u.deg,
    lat=OBSERVATORY_LATITUDE_DEG * u.deg,
    height=OBSERVATORY_ALTITUDE_M * u.m,
)

measurements = [
    load_magzero_measurement(
        path=path,
        location=location,
        zeropoint_key=ZEROPOINT_KEY,
    )
    for path in MAGZERO_FILES
]

measurements_table = pd.DataFrame(measurements).sort_values(
    "airmass_plane_parallel"
).reset_index(drop=True)

measurements_table


### Airmass graph

$$X=1/\sin h$$


In [ ]:
minimum_altitude = max(
    5.0,
    float(measurements_table["altitude_deg"].min()) - 5.0,
)

altitude_grid = np.linspace(minimum_altitude, 90.0, 600)
airmass_grid = 1.0 / np.sin(np.deg2rad(altitude_grid))

plt.figure(figsize=(8, 5.5))
plt.plot(
    altitude_grid,
    airmass_grid,
    label=r"$X=1/\sin(h)=1/\cos(z)$",
)
plt.scatter(
    measurements_table["altitude_deg"],
    measurements_table["airmass_plane_parallel"],
    s=60,
    label="MAGZERO observations",
)

for _, row in measurements_table.iterrows():
    label = row["object_name"] or row["file"]
    plt.annotate(
        str(label),
        (row["altitude_deg"], row["airmass_plane_parallel"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

plt.xlabel("Altitude / elevation h [deg]")
plt.ylabel("Airmass X")
plt.title("Airmass formula check")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

airmass_plot_path = OUTPUT_DIRECTORY / "airmass_formula_check.png"
plt.savefig(airmass_plot_path, dpi=200)
plt.show()


### Atmospheric transmission estimation


In [ ]:
fit_result = fit_zeropoint_vs_airmass(
    table=measurements_table,
    use_scatter_weights=USE_SCATTER_WEIGHTS,
)

fit_result


In [ ]:
p = fit_result["transmission_per_airmass"]
k = fit_result["extinction_mag_per_airmass"]

print(f"Number of files: {len(measurements_table)}")
print(
    "Airmass range: "
    f"{measurements_table['airmass_plane_parallel'].min():.5f} - "
    f"{measurements_table['airmass_plane_parallel'].max():.5f}"
)
print(f"Zenith zeropoint: {fit_result['zenith_zeropoint']:.6f}")
print(f"Extinction coefficient: {k:.6f} mag/airmass")
print(f"Atmospheric transmission p: {p:.6f}")
print(f"Transmission loss per airmass: {(1-p)*100:.3f} %")
print(f"Fit RMSE: {fit_result['rmse_mag']:.6f} mag")

if len(measurements_table) == 2:
    print(
        "\n注意: 2点では直線が一意に決まるため、"
        "フィット不確かさは評価できません。"
    )

if not (0 < p <= 1):
    print(
        "\n警告: 推定透過率が0 < p <= 1の範囲外です。"
        "雲、光学設定、フィルター、時刻差、airmass範囲を確認してください。"
    )


### Zeropoint - Airmass

In [ ]:
x = measurements_table["airmass_plane_parallel"].to_numpy(dtype=float)
y = measurements_table["zeropoint"].to_numpy(dtype=float)

x_grid = np.linspace(
    max(1.0, float(x.min()) - 0.03),
    float(x.max()) + 0.03,
    300,
)
y_grid = (
    fit_result["zenith_zeropoint"]
    + fit_result["slope_mag_per_airmass"] * x_grid
)

plt.figure(figsize=(8, 5.5))

if measurements_table["zeropoint_scatter"].notna().all():
    plt.errorbar(
        x,
        y,
        yerr=measurements_table["zeropoint_scatter"],
        fmt="o",
        capsize=3,
        label="Measured zeropoints",
    )
else:
    plt.scatter(x, y, s=60, label="Measured zeropoints")

plt.plot(
    x_grid,
    y_grid,
    linestyle="--",
    label=(
        f"Linear fit\n"
        f"p = {p:.5f}, k = {k:.4f} mag/airmass"
    ),
)

for _, row in measurements_table.iterrows():
    label = row["object_name"] or row["file"]
    plt.annotate(
        str(label),
        (row["airmass_plane_parallel"], row["zeropoint"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

plt.xlabel(r"Airmass $X=1/\sin(h)$")
plt.ylabel("Photometric zeropoint [mag]")
plt.title("Photometric zeropoint versus airmass")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

fit_plot_path = OUTPUT_DIRECTORY / "zeropoint_vs_airmass.png"
plt.savefig(fit_plot_path, dpi=200)
plt.show()


### Save result


In [ ]:
measurements_table["zeropoint_fit"] = (
    fit_result["zenith_zeropoint"]
    + fit_result["slope_mag_per_airmass"]
    * measurements_table["airmass_plane_parallel"]
)

measurements_table["zeropoint_residual"] = (
    measurements_table["zeropoint"]
    - measurements_table["zeropoint_fit"]
)

measurements_table["line_of_sight_transmission"] = (
    p ** measurements_table["airmass_plane_parallel"]
)

csv_output_path = (
    OUTPUT_DIRECTORY / "atmospheric_transmission_measurements.csv"
)
json_output_path = (
    OUTPUT_DIRECTORY / "atmospheric_transmission_fit.json"
)

measurements_table.to_csv(csv_output_path, index=False)

output_data = {
    "site": {
        "latitude_deg": OBSERVATORY_LATITUDE_DEG,
        "longitude_deg": OBSERVATORY_LONGITUDE_DEG,
        "altitude_m": OBSERVATORY_ALTITUDE_M,
    },
    "zeropoint_key": ZEROPOINT_KEY,
    "airmass_definition": {
        "altitude": "X = 1 / sin(h)",
        "zenith_angle": "X = 1 / cos(z)",
        "relation": "z = 90 deg - h",
    },
    "fit": {
        key: (
            None
            if isinstance(value, float) and not np.isfinite(value)
            else value
        )
        for key, value in fit_result.items()
    },
    "measurements": measurements_table.to_dict(orient="records"),
}

with json_output_path.open("w", encoding="utf-8") as file:
    json.dump(output_data, file, ensure_ascii=False, indent=2, default=str)

print(csv_output_path)
print(json_output_path)
print(airmass_plot_path)
print(fit_plot_path)

measurements_table
